# WTI Crude Oil Price Analysis & Forecasting
## Macroeconomic Associations, Leakage-Free Validation, and Time-Series Forecasting

**Author:** Sai Kamala Priya Areti  
**Checked-in snapshot:** September 14, 2015 to May 25, 2026  

### Research question
**How are macroeconomic and financial-market conditions associated with movements in WTI crude oil prices, and how much explanatory or predictive information do they add beyond recent oil-price dynamics?**

This notebook deliberately separates **explanatory analysis** from **forecasting**. It uses time-ordered validation, a persistence benchmark, and next-week targets so that rolling WTI features do not leak the outcome being predicted.

## 1. Setup and frozen data snapshots

The repository includes frozen weekly-market and monthly-macro snapshots so the reported results remain reproducible even if market-data vendors revise historical observations.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

ROOT = Path('.')
DATA = ROOT / 'data'
CHARTS = ROOT / 'charts'
CHARTS.mkdir(exist_ok=True)

market = pd.read_csv(DATA / 'market_weekly_snapshot.csv', parse_dates=['Date']).set_index('Date')
macro = pd.read_csv(DATA / 'macro_monthly_snapshot.csv', parse_dates=['Date']).set_index('Date')

print(f'Weekly market snapshot: {len(market):,} rows | {market.index.min().date()} to {market.index.max().date()}')
print(f'Monthly macro snapshot: {len(macro):,} rows | {macro.index.min().date()} to {macro.index.max().date()}')

Weekly market snapshot: 559 rows | 2015-09-14 to 2026-05-25
Monthly macro snapshot: 129 rows | 2015-09-01 to 2026-05-01


### Data-quality checks

The checks below enforce chronological ordering, unique dates, and complete weekly market inputs. A missing monthly CPI observation may remain missing rather than being silently invented for econometric analysis.

In [2]:
assert market.index.is_monotonic_increasing
assert macro.index.is_monotonic_increasing
assert not market.index.duplicated().any()
assert not macro.index.duplicated().any()
assert market[['WTI','DXY','VIX','SP500','GOLD','NATGAS']].notna().all().all()

print('Duplicate weekly dates:', int(market.index.duplicated().sum()))
print('Missing weekly market values:', int(market.isna().sum().sum()))
print('Missing monthly macro values:')
print(macro.isna().sum())

Duplicate weekly dates: 0
Missing weekly market values: 0
Missing monthly macro values:
CPI         1
FED_RATE    0
dtype: int64


## 2. Descriptive view

Price-level correlations among trending time series can be misleading, so the correlation analysis below uses **WTI dollar changes** and **weekly percentage returns** for the other market variables.

In [3]:
changes = pd.DataFrame(index=market.index)
changes['WTI change'] = market['WTI'].diff()
for col, label in [('DXY','DXY return'),('VIX','VIX return'),('SP500','S&P 500 return'),('GOLD','Gold return'),('NATGAS','Natural gas return')]:
    changes[label] = market[col].pct_change() * 100

corr = changes.dropna().corr()
print(corr.round(3))

                    WTI change  DXY return  ...  Gold return  Natural gas return
WTI change               1.000       0.030  ...        0.055               0.056
DXY return               0.030       1.000  ...       -0.418              -0.068
VIX return              -0.161       0.031  ...        0.008              -0.055
S&P 500 return           0.128      -0.248  ...        0.065               0.053
Gold return              0.055      -0.418  ...        1.000               0.097
Natural gas return       0.056      -0.068  ...        0.097               1.000

[6 rows x 6 columns]


## 3. Weekly explanatory regression

The weekly specification asks whether financial-market movements explain WTI changes **in addition to** WTI's own recent movement. OLS is estimated with HAC/Newey-West standard errors.

In [4]:
wreg = pd.DataFrame(index=market.index)
wreg['WTI_change'] = market['WTI'].diff()
wreg['WTI_change_lag1'] = wreg['WTI_change'].shift(1)
for c in ['DXY','VIX','SP500','GOLD','NATGAS']:
    wreg[f'{c}_ret'] = market[c].pct_change() * 100
wreg = wreg.dropna()

def ols_hac(y, X, maxlags):
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})

weekly_base = ols_hac(wreg['WTI_change'], wreg[['WTI_change_lag1']], 4)
weekly_full = ols_hac(
    wreg['WTI_change'],
    wreg[['WTI_change_lag1','DXY_ret','VIX_ret','SP500_ret','GOLD_ret','NATGAS_ret']],
    4
)

print(f'Baseline R²: {weekly_base.rsquared:.3f}')
print(f'Financial extension R²: {weekly_full.rsquared:.3f}')
print(f'Lagged WTI-change coefficient: {weekly_full.params["WTI_change_lag1"]:.3f}')
print(f'Lagged WTI-change HAC p-value: {weekly_full.pvalues["WTI_change_lag1"]:.4f}')

Baseline R²: 0.060
Financial extension R²: 0.091
Lagged WTI-change coefficient: -0.235
Lagged WTI-change HAC p-value: 0.0205


**Interpretation:** the negative lag coefficient is consistent with short-run mean reversion in this sample. The financial variables increase explanatory power only modestly. These are associations, not causal estimates.

## 4. Monthly macroeconomic extension

CPI and the Effective Federal Funds Rate are monthly series. They are therefore analyzed at monthly frequency rather than repeated across weekly observations.

In [5]:
monthly_market = market.resample('MS').last()
monthly = monthly_market.join(macro, how='inner')

mreg = pd.DataFrame(index=monthly.index)
mreg['WTI_change'] = monthly['WTI'].diff()
mreg['WTI_change_lag1'] = mreg['WTI_change'].shift(1)
for c in ['DXY','VIX','SP500','GOLD','NATGAS']:
    mreg[f'{c}_ret'] = monthly[c].pct_change() * 100
mreg['CPI_inflation'] = monthly['CPI'].pct_change(fill_method=None) * 100
mreg['FED_change'] = monthly['FED_RATE'].diff()
mreg = mreg.dropna()

monthly_full = ols_hac(
    mreg['WTI_change'],
    mreg[['WTI_change_lag1','DXY_ret','VIX_ret','SP500_ret','GOLD_ret','NATGAS_ret','CPI_inflation','FED_change']],
    3
)

print(f'Monthly R²: {monthly_full.rsquared:.3f}')
print(f'Monthly adjusted R²: {monthly_full.rsquared_adj:.3f}')
print(f'CPI inflation coefficient: {monthly_full.params["CPI_inflation"]:.3f}')
print(f'CPI HAC p-value: {monthly_full.pvalues["CPI_inflation"]:.6f}')

Monthly R²: 0.266
Monthly adjusted R²: 0.216
CPI inflation coefficient: 13.465
CPI HAC p-value: 0.000023


The CPI result is **contemporaneous** and should not be interpreted as proof that inflation causes oil-price movements. Oil prices can also affect inflation, and both can react to common economic shocks.

## 5. Leakage-free one-week-ahead XGBoost

The target is **next week's WTI price**. Features are constructed using information at the current week or earlier. This avoids the earlier pitfall of using a rolling average containing the target week's WTI while predicting that same week's price.

In [6]:
df = market.copy()
for lag in [1,2,4,8,12]:
    df[f'WTI_lag{lag}'] = df['WTI'].shift(lag)
for window in [4,12,26]:
    df[f'WTI_ma{window}'] = df['WTI'].rolling(window).mean()
df['WTI_momentum4'] = df['WTI'] - df['WTI'].shift(4)
for c in ['DXY','VIX','SP500','GOLD','NATGAS']:
    df[f'{c}_pct'] = df[c].pct_change()
df['DXY_ma4'] = df['DXY'].rolling(4).mean()
df['VIX_ma4'] = df['VIX'].rolling(4).mean()
df['month'] = df.index.month
df['quarter'] = df.index.quarter

df['target_next_WTI'] = df['WTI'].shift(-1)
df['target_date'] = pd.Series(df.index, index=df.index).shift(-1)
model_df = df.dropna().copy()

features = [
    'WTI','DXY','VIX','SP500','GOLD','NATGAS',
    'WTI_lag1','WTI_lag2','WTI_lag4','WTI_lag8','WTI_lag12',
    'WTI_ma4','WTI_ma12','WTI_ma26','WTI_momentum4',
    'DXY_ma4','VIX_ma4','DXY_pct','VIX_pct','SP500_pct','GOLD_pct','NATGAS_pct',
    'month','quarter'
]

assert (model_df['target_date'] > model_df.index).all()

xgb_args = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    objective='reg:squarederror'
)

### Chronological 80/20 holdout

The model is compared with a persistence forecast: **next week's WTI = this week's WTI**.

In [7]:
split = int(len(model_df) * 0.8)
train = model_df.iloc[:split]
test = model_df.iloc[split:]

model = XGBRegressor(**xgb_args)
model.fit(train[features], train['target_next_WTI'])
pred = model.predict(test[features])
persistence = test['WTI'].values

print(f'XGBoost R²: {r2_score(test["target_next_WTI"], pred):.3f}')
print(f'XGBoost MAE: ${mean_absolute_error(test["target_next_WTI"], pred):.2f}')
print(f'Persistence R²: {r2_score(test["target_next_WTI"], persistence):.3f}')
print(f'Persistence MAE: ${mean_absolute_error(test["target_next_WTI"], persistence):.2f}')

XGBoost R²: 0.850
XGBoost MAE: $3.15
Persistence R²: 0.841
Persistence MAE: $3.08


For the checked-in snapshot, XGBoost does not improve MAE over the persistence benchmark on this holdout. That result is retained rather than hidden because benchmark comparison is part of responsible model evaluation.

### Stricter 2023–2024 out-of-time backtest

In [8]:
train_bt = model_df[model_df['target_date'] < '2023-01-01']
test_bt = model_df[(model_df['target_date'] >= '2023-01-01') & (model_df['target_date'] < '2025-01-01')]

model_bt = XGBRegressor(**xgb_args)
model_bt.fit(train_bt[features], train_bt['target_next_WTI'])
pred_bt = model_bt.predict(test_bt[features])
persistence_bt = test_bt['WTI'].values

print(f'XGBoost backtest R²: {r2_score(test_bt["target_next_WTI"], pred_bt):.3f}')
print(f'XGBoost backtest MAE: ${mean_absolute_error(test_bt["target_next_WTI"], pred_bt):.2f}')
print(f'Persistence backtest R²: {r2_score(test_bt["target_next_WTI"], persistence_bt):.3f}')
print(f'Persistence backtest MAE: ${mean_absolute_error(test_bt["target_next_WTI"], persistence_bt):.2f}')

XGBoost backtest R²: 0.572
XGBoost backtest MAE: $2.89
Persistence backtest R²: 0.689
Persistence backtest MAE: $2.55


![Leakage-free backtest](charts/backtest_final.png)

### Feature importance

Feature importance is reported as **relative XGBoost model importance**. It is not interpreted as percent of economic variance explained or as evidence of causality.

![Corrected feature importance](charts/feature_importance_final.png)

## 6. Prophet forecast snapshot

Prophet is used as a separate time-series forecasting exercise. The saved chart corresponds to a 26-week point forecast generated from WTI data through May 25, 2026.

![Prophet forecast](charts/prophet_forecast_2026.png)

In [ ]:
# Optional: regenerate the Prophet forecast from the frozen snapshot.
# Install dependencies first with: pip install -r requirements.txt
try:
    from prophet import Prophet

    prophet_df = market[['WTI']].reset_index()
    prophet_df.columns = ['ds', 'y']
    model_prophet = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        interval_width=0.80,
    )
    model_prophet.fit(prophet_df)
    future = model_prophet.make_future_dataframe(periods=26, freq='W')
    forecast = model_prophet.predict(future)
    future_only = forecast[forecast['ds'] > prophet_df['ds'].max()]
    display(future_only[['ds','yhat','yhat_lower','yhat_upper']].head())
except ImportError:
    print('Prophet is not installed. Run: pip install -r requirements.txt')

## 7. Optional live refresh

The frozen snapshots above are used for reproducibility. This optional block shows how to refresh the market and FRED sources. Re-running with newer data can change model metrics, which should then be updated in the README and Tableau dashboard.

In [10]:
# Optional live refresh; requires internet access.
# import yfinance as yf
#
# START = '2015-09-01'
# END = '2026-05-26'  # end is exclusive in yfinance
# tickers = {
#     'WTI':'CL=F', 'DXY':'DX-Y.NYB', 'VIX':'^VIX',
#     'SP500':'^GSPC', 'GOLD':'GC=F', 'NATGAS':'NG=F'
# }
#
# market_data = {}
# for name, ticker in tickers.items():
#     temp = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
#     close = temp['Close']
#     if isinstance(close, pd.DataFrame):
#         close = close.iloc[:, 0]
#     market_data[name] = close
# refreshed_market = pd.DataFrame(market_data).resample('W-MON').last().dropna()
#
# def load_fred_csv(series_id):
#     url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
#     frame = pd.read_csv(url)
#     date_col = 'observation_date' if 'observation_date' in frame.columns else frame.columns[0]
#     frame[date_col] = pd.to_datetime(frame[date_col])
#     frame[series_id] = pd.to_numeric(frame[series_id], errors='coerce')
#     return frame.set_index(date_col)[series_id]
#
# refreshed_macro = pd.concat({
#     'CPI': load_fred_csv('CPIAUCSL'),
#     'FED_RATE': load_fred_csv('FEDFUNDS')
# }, axis=1)
#
# refreshed_market.to_csv(DATA / 'market_weekly_snapshot.csv')
# refreshed_macro.loc[START:END].to_csv(DATA / 'macro_monthly_snapshot.csv')

## 8. Limitations

- Association is not causation.
- April 2020 contains an extreme negative front-month WTI futures observation that can materially influence estimates.
- CPI and the Effective Federal Funds Rate are monthly and have release timing; they are not treated as new weekly information in the XGBoost forecast.
- XGBoost feature importance measures model contribution, not economic causality.
- Oil-price forecasts are sensitive to structural breaks, geopolitical shocks, and the evaluation window. A simple persistence benchmark remains difficult to beat consistently.